# 22 — Agentic RAG

> **Tier 4 | Agentic & Self-Improving**

## What is Agentic RAG?

Previous patterns (Corrective, Self, Iterative, Recursive) follow a **fixed retrieval
strategy** decided at design time. **Agentic RAG** hands control to the LLM itself:
the agent is given a set of retrieval tools and decides — at runtime, per question —
*which tool to call, with what arguments, and how many times*.

### Tool set

| Tool | When the agent uses it |
|------|------------------------|
| `vector_search(query, top_k)` | Semantic questions — "how does X work?" |
| `keyword_search(keyword, top_k)` | Exact terms — acronyms, proper nouns, model names |
| `section_lookup(section_title)` | When it knows which section to target |
| `list_sections()` | When it needs to discover what topics are available |

The agent may call **any combination** of tools in any order before generating
its final answer — or call none at all if it determines the question is
unanswerable from the KB.

## PDF Framework: pymupdf (fitz) — block-type extraction

This notebook uses **pymupdf** with `page.get_text("blocks")` which returns
each text block with a `type` flag (`0` = text, `1` = image). We use this to:

- Skip image-only blocks automatically
- Extract `block_no` per chunk → stored as metadata → enables `section_lookup`
  to do positional filtering without a separate title-detection pass

| Feature | NB 19 pymupdf use | NB 22 pymupdf use |
|---------|------------------|------------------|
| API | `get_text("dict")` — span/line level | `get_text("blocks")` — block level |
| Font info | Yes (size, bold flags) | Not needed |
| Block type filter | No | Yes (`type==0` skips image blocks) |
| Purpose | Heading detection | Fast block-level chunking for agent tools |

## Flow Diagram

```mermaid
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pymupdf blocks"]
        PDF(["📄 climate.pdf"])
        PDF --> FZ["fitz.open()\nget_text('blocks')"]
        FZ --> BLK["Text blocks\n(type=0 only)"]
        BLK --> META["Chunk + metadata\n(page, block_no, section)"]
        META --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nagentic_rag_22")]
    end

    subgraph AGENT ["🤖  AGENTIC RETRIEVAL LOOP (Strands)"]
        Q(["❓ Query"])
        Q --> AG["Strands Agent\n+ tool set"]
        AG --> T1["vector_search\n(semantic)"]
        AG --> T2["keyword_search\n(exact match)"]
        AG --> T3["section_lookup\n(by title)"]
        AG --> T4["list_sections\n(discovery)"]
        T1 --> AG
        T2 --> AG
        T3 --> AG
        T4 --> AG
        AG --> FINAL(["✅ Final answer\n(agent decides when done)"])
    end

    QDRANT --> T1
    QDRANT --> T2
    QDRANT --> T3
    QDRANT --> T4

    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style AGENT fill:#fdf4ff,stroke:#9333ea,color:#3b0764
```


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pymupdf blocks"]
        PDF(["📄 climate.pdf"])
        PDF --> FZ["fitz.open()\nget_text('blocks')"]
        FZ --> BLK["Text blocks\n(type=0 only)"]
        BLK --> META["Chunk + metadata\n(page, block_no, section)"]
        META --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nagentic_rag_22")]
    end

    subgraph AGENT ["🤖  AGENTIC RETRIEVAL LOOP (Strands)"]
        Q(["❓ Query"])
        Q --> AG["Strands Agent\n+ tool set"]
        AG --> T1["vector_search\n(semantic)"]
        AG --> T2["keyword_search\n(exact match)"]
        AG --> T3["section_lookup\n(by title)"]
        AG --> T4["list_sections\n(discovery)"]
        T1 --> AG
        T2 --> AG
        T3 --> AG
        T4 --> AG
        AG --> FINAL(["✅ Final answer\n(agent decides when done)"])
    end

    QDRANT --> T1
    QDRANT --> T2
    QDRANT --> T3
    QDRANT --> T4

    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style AGENT fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "strands-agents", "pymupdf", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid
from typing import List, Dict, Optional
from dotenv import load_dotenv

import boto3
import fitz                           # pymupdf
from strands import Agent, tool
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

COLLECTION_NAME = "agentic_rag_22"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
DEFAULT_TOP_K   = 5

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF        : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")


## Step 3 — Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name = collection_name
        self._chunks: List[Dict] = []   # in-memory index for keyword/section search
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}'); return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}'); return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
        if exists and force_recreate:
            self._qdrant.delete_collection(self.name); exists = False
        if not exists:
            self._qdrant.create_collection(
                self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
            print(f'Created "{self.name}" (dim={dim})')
        else:
            print(f'"{self.name}" already exists')

    def upsert(self, docs: List[Dict]):
        pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                           payload={'text': d['text'], 'metadata': d.get('metadata', {})})
               for d in docs]
        self._qdrant.upsert(collection_name=self.name, points=pts)
        self._chunks.extend(docs)   # keep plain-text copy for keyword/section ops

    def vector_search(self, qvec: List[float], top_k: int = 5) -> List[Dict]:
        resp = self._qdrant.query_points(
            collection_name=self.name, query=qvec,
            limit=top_k, with_payload=True)
        return [{'text': p.payload.get('text',''),
                 'metadata': p.payload.get('metadata',{}),
                 'score': p.score} for p in resp.points]

    def keyword_search(self, keyword: str, top_k: int = 5) -> List[Dict]:
        kw = keyword.lower()
        hits = [c for c in self._chunks if kw in c['text'].lower()]
        hits.sort(key=lambda c: -c['text'].lower().count(kw))
        return [{'text': h['text'], 'metadata': h.get('metadata',{})}
                for h in hits[:top_k]]

    def section_search(self, section_title: str, top_k: int = 5) -> List[Dict]:
        st = section_title.lower()
        hits = [c for c in self._chunks
                if st in c.get('metadata',{}).get('section','').lower()
                or st in c['text'].lower()[:120]]
        return [{'text': h['text'], 'metadata': h.get('metadata',{})}
                for h in hits[:top_k]]

    def list_sections(self) -> List[str]:
        seen, out = set(), []
        for c in self._chunks:
            s = c.get('metadata',{}).get('section','')
            if s and s not in seen:
                seen.add(s); out.append(s)
        return out

    def count(self) -> int:
        return self._qdrant.get_collection(self.name).points_count or 0

print("VectorStore defined.")


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

bedrock_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

test_emb = embed_text("agentic rag pymupdf blocks test")
print(f"Embedding OK — dim={len(test_emb)}")
print("BedrockModel ready.")


## Step 5 — PDF Loading with pymupdf (`get_text("blocks")`)

`get_text("blocks")` returns `(x0, y0, x1, y1, text, block_no, block_type)` tuples.
`block_type == 0` is text; `block_type == 1` is an image — we skip images automatically.
We use `block_no` as a positional index per page, and the first line of each block
as a section-title signal stored in chunk metadata.


In [ ]:
def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        end = start + size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def load_pdf_pymupdf_blocks(path: str):
    """
    Extracts chunks using pymupdf get_text('blocks').
    Each block is chunked; first line of block used as section title hint.
    Returns (chunks, stats).
    """
    doc    = fitz.open(path)
    chunks = []
    section_titles = []   # ordered list of distinct section hints

    for page_num in range(len(doc)):
        page   = doc[page_num]
        blocks = page.get_text("blocks")   # list of (x0,y0,x1,y1,text,block_no,block_type)

        for block in blocks:
            x0, y0, x1, y1, text, block_no, block_type = block
            if block_type != 0:   # skip image blocks
                continue
            text = text.strip()
            if not text:
                continue

            # Use first line as a section hint
            first_line = text.split('\n')[0].strip()[:80]
            if first_line and first_line not in section_titles:
                section_titles.append(first_line)
            section = first_line

            for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
                chunks.append({
                    'text'     : sub,
                    'page'     : page_num + 1,
                    'block_no' : block_no,
                    'section'  : section,
                    'char_count': len(sub),
                })

    doc.close()
    stats = {
        'n_pages'    : len(doc),
        'n_chunks'   : len(chunks),
        'n_sections' : len(section_titles),
        'avg_chars'  : sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats, section_titles


t0 = time.time()
chunks, stats, section_titles = load_pdf_pymupdf_blocks(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pymupdf blocks extraction : {elapsed:.0f}ms")
print(f"Pages                     : {stats['n_pages']}")
print(f"Chunks                    : {stats['n_chunks']}")
print(f"Section hints             : {stats['n_sections']}")
print(f"Avg chunk length          : {stats['avg_chars']} chars")
print(f"\nFirst 8 section hints:")
for s in section_titles[:8]:
    print(f"  • {s}")


## Step 6 — Connect & Index

In [ ]:
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL, qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL, region=AWS_REGION)
print(f"Backend: {vs._backend}")

vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

docs = [
    {
        'text'     : chunks[i]['text'],
        'embedding': embs[i],
        'metadata' : {
            'page'      : chunks[i]['page'],
            'block_no'  : chunks[i]['block_no'],
            'section'   : chunks[i]['section'],
            'char_count': chunks[i]['char_count'],
            'source'    : 'climate.pdf',
            'pdf_lib'   : 'pymupdf-blocks',
        }
    }
    for i in range(len(chunks))
]
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors in {time.time()-t0:.1f}s")
print(f"Sections available for section_lookup: {len(vs.list_sections())}")


## Step 7 — Strands Tool Definitions

Each tool is decorated with `@tool` so Strands exposes it to the LLM's tool-use loop.
The docstring becomes the tool description the model sees when deciding which to call.


In [ ]:
@tool
def vector_search(query: str, top_k: int = DEFAULT_TOP_K) -> str:
    """Search the knowledge base using semantic similarity.
    Use for conceptual questions like 'how does X work', 'what is Y', 'explain Z'.
    Returns the most relevant passages ranked by similarity score.
    """
    hits = vs.vector_search(embed_text(query), top_k=top_k)
    if not hits:
        return "No results found."
    parts = []
    for i, h in enumerate(hits, 1):
        meta = h['metadata']
        parts.append(
            f"[{i}] page={meta.get('page','?')} score={h['score']:.3f}\n{h['text']}")
    return '\n\n'.join(parts)


@tool
def keyword_search(keyword: str, top_k: int = DEFAULT_TOP_K) -> str:
    """Search for chunks containing an exact keyword or phrase.
    Use for proper nouns, acronyms, model names, or technical terms
    where exact match matters more than semantic similarity.
    """
    hits = vs.keyword_search(keyword, top_k=top_k)
    if not hits:
        return f"No chunks contain the keyword '{keyword}'."
    parts = []
    for i, h in enumerate(hits, 1):
        meta = h['metadata']
        parts.append(
            f"[{i}] page={meta.get('page','?')} section='{meta.get('section','')[:50]}'\n"
            f"{h['text']}")
    return '\n\n'.join(parts)


@tool
def section_lookup(section_title: str, top_k: int = DEFAULT_TOP_K) -> str:
    """Retrieve chunks from a specific section by its title.
    Use when you know (or suspect) which section of the document contains the answer.
    Call list_sections() first if you are unsure of exact section names.
    """
    hits = vs.section_search(section_title, top_k=top_k)
    if not hits:
        return f"No section matching '{section_title}' found. Try list_sections()."
    parts = []
    for i, h in enumerate(hits, 1):
        meta = h['metadata']
        parts.append(
            f"[{i}] page={meta.get('page','?')} section='{meta.get('section','')[:50]}'\n"
            f"{h['text']}")
    return '\n\n'.join(parts)


@tool
def list_sections() -> str:
    """List all section titles available in the knowledge base.
    Use this first when you are unsure what topics the document covers,
    or before calling section_lookup() to confirm the correct section name.
    """
    sections = vs.list_sections()
    if not sections:
        return "No section information available."
    return "Available sections:\n" + '\n'.join(f"  • {s}" for s in sections[:40])


print("Tools defined: vector_search, keyword_search, section_lookup, list_sections")


## Step 8 — Agentic RAG Pipeline

The Strands `Agent` receives the system prompt + tool set. It autonomously:
1. Decides which tools to call (and with what arguments)
2. Processes tool results
3. Calls more tools if needed
4. Generates a final answer when satisfied

We wrap it to capture the tool-call trace for inspection.


In [ ]:
AGENT_SYSTEM = """You are a knowledgeable assistant with access to a weather and climate knowledge base.

You have four retrieval tools:
- vector_search(query)   — semantic similarity search; use for conceptual questions
- keyword_search(keyword) — exact keyword/phrase match; use for acronyms or proper nouns
- section_lookup(title)  — fetch passages from a named section
- list_sections()        — discover what sections exist in the knowledge base

Strategy:
1. For broad questions, start with vector_search.
2. For questions about specific terms or acronyms, also run keyword_search.
3. If you are unsure which section covers the topic, call list_sections() first.
4. Combine evidence from multiple tools when needed.
5. Answer only from the retrieved evidence. If information is not found, say so.
6. Be concise and precise in your final answer.
"""

rag_tools = [vector_search, keyword_search, section_lookup, list_sections]


def agentic_rag(question: str, verbose: bool = True) -> Dict:
    t0    = time.time()
    agent = Agent(
        model=bedrock_model,
        system_prompt=AGENT_SYSTEM,
        tools=rag_tools,
    )

    if verbose:
        print(f"Q: {question}\n")

    answer  = str(agent(question))
    latency = (time.time() - t0) * 1000

    # Extract tool-use trace from message history
    tool_calls = []
    for msg in agent.messages:
        if isinstance(msg.get('content'), list):
            for block in msg['content']:
                if isinstance(block, dict) and block.get('type') == 'tool_use':
                    tool_calls.append({
                        'tool' : block.get('name'),
                        'input': block.get('input', {}),
                    })

    if verbose:
        print(f"\nTool calls ({len(tool_calls)}):")
        for tc in tool_calls:
            inp_str = json.dumps(tc['input'])[:80]
            print(f"  → {tc['tool']}({inp_str})")
        print(f"\nAnswer:\n{answer}")
        print(f"\nLatency: {latency:.0f}ms")
        print("-" * 70)

    return {
        'question'   : question,
        'answer'     : answer,
        'tool_calls' : tool_calls,
        'n_tools'    : len(tool_calls),
        'latency_ms' : latency,
    }


# Demo
result = agentic_rag("What is ensemble forecasting and why is it used?")


## Step 9 — Multi-Question Demo

Questions are chosen to provoke different tool strategies:
- Conceptual → `vector_search`
- Acronym/term → `keyword_search`
- Unknown topic → `list_sections` + `section_lookup`
- Multi-aspect → combination of tools


In [ ]:
test_questions = [
    # Conceptual — vector_search expected
    "How do numerical weather prediction models work?",
    # Acronym — keyword_search expected
    "What does NWP stand for and what are its limitations?",
    # Discovery — list_sections + section_lookup expected
    "What topics does this document cover about climate observation?",
    # Multi-aspect — multiple tools expected
    "Compare satellite-based and ground-based weather observation methods.",
]

all_results = []
for q in test_questions:
    r = agentic_rag(q, verbose=False)
    all_results.append(r)

print(f"{'Question':<55}  {'Tools':>5}  {'ms':>7}")
print("-" * 70)
for r in all_results:
    tool_names = ', '.join(dict.fromkeys(tc['tool'] for tc in r['tool_calls']))
    print(f"{r['question'][:54]:<55}  {r['n_tools']:>5}  {r['latency_ms']:>7.0f}")
    print(f"  tools used: {tool_names or '(none)'}")


## Step 10 — Tool Selection Analysis

Show which tools the agent chose per question and why that makes sense.


In [ ]:
from collections import Counter

print("Tool selection breakdown across all questions:")
print()

all_tool_counts = Counter()
for r in all_results:
    for tc in r['tool_calls']:
        all_tool_counts[tc['tool']] += 1

print(f"{'Tool':<20}  {'Total calls':>11}")
print("-" * 34)
for tool_name, count in sorted(all_tool_counts.items(), key=lambda x: -x[1]):
    print(f"{tool_name:<20}  {count:>11}")

print()
print("Per-question tool trace:")
for r in all_results:
    print(f"\n  Q: {r['question'][:70]}")
    if not r['tool_calls']:
        print("     (no tools called)")
    for i, tc in enumerate(r['tool_calls'], 1):
        inp = json.dumps(tc['input'])[:70]
        print(f"     {i}. {tc['tool']}({inp})")


## Step 11 — Summary

### What Agentic RAG does differently

| Dimension | Fixed-strategy RAG | Agentic RAG |
|-----------|------------------|-------------|
| Retrieval strategy | Predetermined at design time | Decided by LLM at runtime per question |
| Tool variety | One retrieval method | 4 tools: semantic, keyword, section, discovery |
| Adaptability | Same path for every query | Multi-tool combinations for complex queries |
| Control | Engineer controls flow | LLM controls flow |
| Transparency | Predictable | Must inspect tool-call trace to understand behaviour |
| Latency | Lower (1 retrieve call) | Higher (multiple LLM + tool turns) |

### pymupdf block-level API

| Method | Returns | Used for |
|--------|---------|----------|
| `get_text("blocks")` | `(x0,y0,x1,y1, text, block_no, block_type)` | Fast block extraction |
| `block_type == 0` | Text block | Keep |
| `block_type == 1` | Image block | Skip automatically |
| `block_no` | Block index within page | Positional metadata → `section_lookup` |

### PDF Framework Progression

| Notebook | Pattern | Framework | Key capability leveraged |
|----------|---------|-----------|--------------------------|
| 18 | Corrective RAG | **pdfplumber** | `bbox_density` as grader signal |
| 19 | Self RAG | **pymupdf** `get_text("dict")` | Font size/bold → `is_heading` |
| 20 | Iterative RAG | **pdfminer.six** | `LTTextBox.y0` → layout zone tagging |
| 21 | Recursive RAG | **pdfminer.six** heuristic classifier | Element-type hierarchy (Title/NarrativeText…) |
| 22 | Agentic RAG | **pymupdf** `get_text("blocks")` | Block-type filter, block_no positional index |
| 23 | Speculative RAG | **pdfplumber** | Word-level bbox density for hypothesis scoring |

> **Next: [23 — Speculative RAG](../tier4_agentic/23_Speculative_RAG.ipynb)** — generate multiple answer hypotheses, score each against retrieved evidence, return the best-supported one.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {vs.count()} vectors indexed")
print(f"PDF framework : pymupdf get_text('blocks') — block_type filter, block_no index")
print(f"RAG pattern   : Agentic RAG — LLM autonomously selects tools at runtime")
print(f"Tools         : vector_search, keyword_search, section_lookup, list_sections")
print()
print("Notebook 22 — Agentic RAG complete.")
